In [55]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

1. Get Pre-tokenized Word Frequency

In [56]:
from collections import defaultdict

def get_word_freqs(corpus):
    word_freqs = defaultdict(int)
    for text in corpus:
        # pretokenize the sentence
        words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
        # capture only the word
        new_words = [word for word, offset in words_with_offsets]
        for word in new_words:
            word_freqs[word] += 1
    return word_freqs


In [57]:
word_freqs = get_word_freqs(corpus)
word_freqs

defaultdict(int,
            {'This': 3,
             'is': 2,
             'the': 1,
             'Hugging': 1,
             'Face': 1,
             'Course': 1,
             '.': 4,
             'chapter': 1,
             'about': 1,
             'tokenization': 1,
             'section': 1,
             'shows': 1,
             'several': 1,
             'tokenizer': 1,
             'algorithms': 1,
             'Hopefully': 1,
             ',': 1,
             'you': 1,
             'will': 1,
             'be': 1,
             'able': 1,
             'to': 1,
             'understand': 1,
             'how': 1,
             'they': 1,
             'are': 1,
             'trained': 1,
             'and': 1,
             'generate': 1,
             'tokens': 1})

2. [WordPiece] Construct Vocab, which is the first alphabet of each word + the rest but padded with '##{rest}'

In [58]:
def get_vocab(word_freqs):
    vocab = set()
    for word in word_freqs.keys():
        print(word)
        # word = word.split('')
        first, rest = word[0], word[1:]
        vocab.add(first)
        if rest:
            for c in rest:
                vocab.add(f'##{c}')
    vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + sorted(list(vocab))
    return vocab


In [59]:
vocab = get_vocab(word_freqs)
vocab

This
is
the
Hugging
Face
Course
.
chapter
about
tokenization
section
shows
several
tokenizer
algorithms
Hopefully
,
you
will
be
able
to
understand
how
they
are
trained
and
generate
tokens


['[PAD]',
 '[UNK]',
 '[CLS]',
 '[SEP]',
 '[MASK]',
 '##a',
 '##b',
 '##c',
 '##d',
 '##e',
 '##f',
 '##g',
 '##h',
 '##i',
 '##k',
 '##l',
 '##m',
 '##n',
 '##o',
 '##p',
 '##r',
 '##s',
 '##t',
 '##u',
 '##v',
 '##w',
 '##y',
 '##z',
 ',',
 '.',
 'C',
 'F',
 'H',
 'T',
 'a',
 'b',
 'c',
 'g',
 'h',
 'i',
 's',
 't',
 'u',
 'w',
 'y']

3. [WordPiece] Splitting each word, with all the letters that are not the first prefixed by '##'

In [60]:
def get_splits(word_freqs):
    splits = {
        word: [c if i == 0 else f'##{c}' for i,c in enumerate(word)]
        for word in word_freqs.keys()
    }
    return splits

In [61]:
splits = get_splits(word_freqs)

for i,key in enumerate(splits.keys()):
    print(f"{key}: {splits[key]}")
    if i >= 3:
        break

This: ['T', '##h', '##i', '##s']
is: ['i', '##s']
the: ['t', '##h', '##e']
Hugging: ['H', '##u', '##g', '##g', '##i', '##n', '##g']


4. Compute pair score from split based on this formula below
    
    $score=\frac{(freq\_of\_pair)}{(freq\_of\_first\_element \times freq\_of\_second\_element)}$

In [62]:
def compute_pair_scores(splits, word_freqs):
    letter_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        word_split = splits[word] # get the split

        # only if there is more than 2 element we go into sliding window mode
        if len(word_split) > 1:
            for i in range(len(word_split) - 1):
                pair = (word_split[i], word_split[i+1])
                pair_freqs[pair] += freq
                letter_freqs[word_split[i]] += freq # notice we only scan up until the 2nd last
        
        # we will just add the last element here
        letter_freqs[word_split[-1]] += freq
    
    scores = defaultdict(float)
    for pair, freq in pair_freqs.items():
        scores[pair] = freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])

    return scores

In [63]:
scores = compute_pair_scores(splits, word_freqs)

for i, key in enumerate(scores.keys()):
    print(f"{key}: {scores[key]}")
    if i == 3:
        break

('T', '##h'): 0.125
('##h', '##i'): 0.03409090909090909
('##i', '##s'): 0.02727272727272727
('i', '##s'): 0.1


4. Finding pair with the best score

In [64]:
def get_best_score(scores):
    best_pair = ""
    max_score = None
    for pair, score in scores.items():
        if max_score is None or score > max_score:
            best_pair = pair
            max_score = score
    return best_pair, max_score

In [65]:
best_pair, max_score = get_best_score(scores)
best_pair, max_score

(('a', '##b'), 0.2)

5. Merge the best scoring pair + add to our vocab as well

In [66]:
def merge_pair(a,b,splits,vocab):
    for word in word_freqs:
        split = splits[word]
        if len(split) <= 1:
            continue
        i = 0
        while i < len(split) - 1:
            left,right = split[i],split[i+1]
            if left == a and right == b:
                merge = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merge] +split[i+2:]
            else:
                i += 1
        splits[word] = split
    
    merge = a + b[2:] if b.startswith("##") else a + b
    vocab.append(merge)
    
    return splits, vocab


In [67]:
# splits, vocab = merge_pair("a","##b",splits,vocab)
# splits["about"]

In [68]:
vocab[-3:]

['u', 'w', 'y']

6. Final vocab construction

In [69]:
vocab_size = 70

while len(vocab) < 70:
    scores = compute_pair_scores(splits, word_freqs)
    best_pair, best_score = get_best_score(scores)
    splits, vocab = merge_pair(best_pair[0], best_pair[1], splits, vocab)

vocab

['[PAD]',
 '[UNK]',
 '[CLS]',
 '[SEP]',
 '[MASK]',
 '##a',
 '##b',
 '##c',
 '##d',
 '##e',
 '##f',
 '##g',
 '##h',
 '##i',
 '##k',
 '##l',
 '##m',
 '##n',
 '##o',
 '##p',
 '##r',
 '##s',
 '##t',
 '##u',
 '##v',
 '##w',
 '##y',
 '##z',
 ',',
 '.',
 'C',
 'F',
 'H',
 'T',
 'a',
 'b',
 'c',
 'g',
 'h',
 'i',
 's',
 't',
 'u',
 'w',
 'y',
 'ab',
 '##fu',
 'Fa',
 'Fac',
 '##ct',
 '##ful',
 '##full',
 '##fully',
 'Th',
 'ch',
 '##hm',
 'cha',
 'chap',
 'chapt',
 '##thm',
 'Hu',
 'Hug',
 'Hugg',
 'sh',
 'th',
 'is',
 '##thms',
 '##za',
 '##zat',
 '##ut']

How to tokenize using the created vocab?

In [74]:
def encode_word(word, vocab):
    tokens = []
    # while the word is still present (can still be chopped chopped)
    while len(word) > 0:
        # we start from lat index
        i = len(word)
        # if its still a valid right-to-left scanning, and the left-to-right counterpart is not in vocab, we decrement
        while i > 0 and word[:i] not in vocab:
            i -= 1
        # if nothing found, we return the [UNK]
        if i == 0:
            return ["[UNK]"]
        # if something is found
        tokens.append(word[:i])
        word = word[i:]
        # since this is subword now, we add ## prefix
        if len(word) > 0:
            word = f"##{word}"
    
    return tokens


# '''
# Right-to-left (greedy longest-match) scanning ensures we find the longest possible subword in vocab at each step (maximal-matching). Left-to-right could miss longer valid subwords, leading to less meaningful splits and inefficient tokenization.
# '''

In [75]:
print(encode_word("Hugging",vocab))
print(encode_word("H0gging",vocab))

['Hugg', '##i', '##n', '##g']
['[UNK]']


In [78]:
def tokenize(text, vocab):
    # split using pre-tokenizer
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    # apply word encoding using the available vocab
    encoded_words = [encode_word(word,vocab) for word in pre_tokenized_text]
    return sum(encoded_words, [])

In [80]:
tokenize("This is the Hugging Face course!",vocab)

['Th',
 '##i',
 '##s',
 'is',
 'th',
 '##e',
 'Hugg',
 '##i',
 '##n',
 '##g',
 'Fac',
 '##e',
 'c',
 '##o',
 '##u',
 '##r',
 '##s',
 '##e',
 '[UNK]']